# Demo: Predict Elastic Properties from RVE Image + Constituent Properties

This notebook demonstrates how to load the trained fusion model and predict
the 9 homogenized elastic constants of a unidirectional composite.

In [ ]:
# Demo: Predict Elastic Properties from RVE Image + Constituent Properties

This notebook demonstrates how to load the trained fusion model and predict
the 9 homogenized elastic constants of a unidirectional composite.

In [ ]:
import numpy as np
import pandas as pd
import pickle
import tensorflow as tf
from tensorflow import keras

TARGET_COLS = ["E11", "E22", "E33", "G12", "G13", "G23", "v12", "v13", "v23"]

# Custom loss functions (must match training)
IDX = {p: TARGET_COLS.index(p) for p in TARGET_COLS}
LAMBDA_PHYSICS = 0.1

def physics_consistency_loss(y_true, y_pred):
    mse = tf.reduce_mean(tf.square(y_true - y_pred))
    diff_E = y_pred[:, IDX['E22']] - y_pred[:, IDX['E33']]
    diff_G = y_pred[:, IDX['G12']] - y_pred[:, IDX['G13']]
    diff_v = y_pred[:, IDX['v12']] - y_pred[:, IDX['v13']]
    physics = tf.reduce_mean(
        tf.square(diff_E) + tf.square(diff_G) + tf.square(diff_v))
    return mse + LAMBDA_PHYSICS * physics

def mse_only(y_true, y_pred):
    return tf.reduce_mean(tf.square(y_true - y_pred))

def physics_penalty_only(y_true, y_pred):
    diff_E = y_pred[:, IDX['E22']] - y_pred[:, IDX['E33']]
    diff_G = y_pred[:, IDX['G12']] - y_pred[:, IDX['G13']]
    diff_v = y_pred[:, IDX['v12']] - y_pred[:, IDX['v13']]
    return tf.reduce_mean(
        tf.square(diff_E) + tf.square(diff_G) + tf.square(diff_v))

custom_objects = {
    "physics_consistency_loss": physics_consistency_loss,
    "mse_only": mse_only,
    "physics_penalty_only": physics_penalty_only,
}

In [ ]:
# ── Load model and scalers ──────────────────────────────────────
# Update these paths to point to your saved files
MODEL_PATH = "fusion_model.keras"
SCALER_TAB_PATH = "scaler_tab.pkl"
SCALER_OUT_PATH = "scaler_output.pkl"

model = keras.models.load_model(MODEL_PATH, custom_objects=custom_objects)

with open(SCALER_TAB_PATH, "rb") as f:
    sc_tab = pickle.load(f)
with open(SCALER_OUT_PATH, "rb") as f:
    sc_out = pickle.load(f)

print(f"Model loaded: {model.count_params():,} parameters")

In [ ]:
# ── Prepare input ───────────────────────────────────────────────
# Example constituent properties (T300/epoxy 7901)
# Order: Em, vm, Ef1, Ef2, Gf12, Gf23, vf12, vf23
sample_properties = np.array([[
    3.17, 0.35, 230.0, 15.0, 15.0, 7.0, 0.20, 0.20
]], dtype=np.float32)

print("Input constituent properties:")
print(f"  Em={sample_properties[0,0]} GPa, vm={sample_properties[0,1]}")
print(f"  Ef1={sample_properties[0,2]} GPa, Ef2={sample_properties[0,3]} GPa")

# Scale tabular input
X_tab = sc_tab.transform(sample_properties).astype(np.float32)

# Create a placeholder 384x384 image
# In practice, replace this with your real RVE cross-section image:
#   img = cv2.imread("my_rve.png", cv2.IMREAD_GRAYSCALE)
#   img = cv2.resize(img, (384, 384), interpolation=cv2.INTER_AREA) / 255.0
#   img = (img - 0.4982)[..., np.newaxis]  # subtract training mean
X_img = np.zeros((1, 384, 384, 1), dtype=np.float32)

print(f"\nImage shape: {X_img.shape}")
print(f"Tabular shape: {X_tab.shape}")

In [ ]:
# ── Predict ─────────────────────────────────────────────────────
y_pred_sc = model.predict(
    {"image_input": X_img, "tabular_input": X_tab}, verbose=0
)
y_pred = sc_out.inverse_transform(y_pred_sc)

# Display results
print("\nPredicted Elastic Constants:")
print("-" * 40)
for prop, val in zip(TARGET_COLS, y_pred[0]):
    unit = "GPa" if prop.startswith(("E", "G")) else ""
    print(f"  {prop:>4} = {val:>10.4f} {unit}")

# Transverse isotropy check
print("\nTransverse Isotropy Check:")
for p1, p2 in [("E22", "E33"), ("G12", "G13"), ("v12", "v13")]:
    i1, i2 = IDX[p1], IDX[p2]
    rd = abs(y_pred[0, i1] - y_pred[0, i2]) / abs(y_pred[0, i1]) * 100
    print(f"  {p1}/{p2}: deviation = {rd:.4f}%")


## Notes
- Replace the placeholder image with a real 384x384 binary RVE cross-section.
- Normalize images to [0,1] and subtract the training mean (0.4982).
- Constituent order: Em, vm, Ef1, Ef2, Gf12, Gf23, vf12, vf23.
- The model expects the custom physics loss to be defined before loading.